In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Tecniche di mitigazione e soppressione degli errori

> **Note:** La versione beta di un nuovo modello di esecuzione è ora disponibile. Il modello di esecuzione diretta offre maggiore flessibilità nella personalizzazione del tuo workflow di mitigazione degli errori. Consulta la guida al [modello di esecuzione diretta](/guides/directed-execution-model) per maggiori informazioni.


<details>
<summary><b>Versioni dei pacchetti</b></summary>

Il codice in questa pagina è stato sviluppato utilizzando i seguenti requisiti.
Si raccomanda di usare queste versioni o versioni più recenti.

```
qiskit-ibm-runtime~=0.43.1
```
</details>

Le tecniche di mitigazione e soppressione degli errori vengono usate per migliorare la qualità dei risultati quando si scala verso carichi di lavoro più grandi. Questa pagina fornisce spiegazioni ad alto livello delle tecniche di soppressione e mitigazione degli errori disponibili tramite Qiskit Runtime.

La seguente cella importa la primitiva Estimator e crea un backend che verrà utilizzato per inizializzare l'Estimator nelle celle di codice successive.

In [1]:
from qiskit_ibm_runtime import EstimatorV2 as Estimator
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend = service.least_busy()

## Disaccoppiamento dinamico
I circuiti quantistici vengono eseguiti sull'hardware IBM&reg; come sequenze di impulsi a microonde che devono essere pianificati ed eseguiti a intervalli di tempo precisi.
Purtroppo, le interazioni indesiderate tra qubit possono causare errori coerenti sui qubit inattivi. Il disaccoppiamento dinamico funziona inserendo sequenze di impulsi sui qubit inattivi per annullare approssimativamente l'effetto di questi errori. Ogni sequenza di impulsi inserita equivale a un'operazione identità, ma la presenza fisica degli impulsi ha l'effetto di sopprimere gli errori.
Esistono molte possibili scelte di sequenze di impulsi, e quale sequenza sia migliore per ogni caso particolare resta un [campo di ricerca attivo](https://journals.aps.org/prapplied/abstract/10.1103/PhysRevApplied.20.064027).

Nota che il disaccoppiamento dinamico è principalmente utile per i circuiti che contengono lacune in cui alcuni qubit rimangono inattivi senza che vi agiscano operazioni. Se le operazioni nel circuito sono molto dense, così che tutti i qubit sono occupati la maggior parte del tempo, l'aggiunta di impulsi di disaccoppiamento dinamico potrebbe non migliorare le prestazioni. In effetti, potrebbe persino peggiorarle a causa delle imperfezioni degli impulsi stessi.

Il diagramma qui sotto illustra il disaccoppiamento dinamico con una sequenza di impulsi XX. Il circuito astratto a sinistra viene mappato su uno schedule di impulsi a microonde in alto a destra. In basso a destra è rappresentato lo stesso schedule, ma con una sequenza di due impulsi X inserita durante un periodo di inattività del primo qubit.

![Rappresentazione del disaccoppiamento dinamico](../docs/images/guides/error-mitigation-explanation/dd.avif)

Il disaccoppiamento dinamico può essere abilitato impostando `enable` a `True` nelle [opzioni di disaccoppiamento dinamico](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-dynamical-decoupling-options). L'opzione `sequence_type` può essere usata per scegliere tra diverse sequenze di impulsi. Il tipo di sequenza predefinito è `"XX"`.

La seguente cella di codice mostra come abilitare il disaccoppiamento dinamico per l'Estimator e scegliere una sequenza di disaccoppiamento dinamico.

In [2]:
estimator = Estimator(mode=backend)
estimator.options.dynamical_decoupling.enable = True
estimator.options.dynamical_decoupling.sequence_type = "XpXm"

## Twirling di Pauli
Il twirling, noto anche come [randomized compiling](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.94.052325), è una tecnica ampiamente utilizzata per convertire canali di rumore arbitrari in canali di rumore con una struttura più specifica.

Il twirling di Pauli è un tipo speciale di twirling che utilizza operazioni di Pauli. Ha l'effetto di trasformare qualsiasi canale quantistico in un canale di Pauli. Eseguito da solo, può mitigare il rumore coerente perché il rumore coerente tende ad accumularsi in modo quadratico con il numero di operazioni, mentre il rumore di Pauli si accumula in modo lineare. Il twirling di Pauli è spesso combinato con altre tecniche di mitigazione degli errori che funzionano meglio con il rumore di Pauli rispetto al rumore arbitrario.

Il twirling di Pauli è implementato racchiudendo un insieme scelto di gate con gate di Pauli a singolo qubit scelti casualmente, in modo tale che l'effetto ideale del gate rimanga lo stesso. Il risultato è che un singolo circuito viene sostituito da un insieme casuale di circuiti, tutti con lo stesso effetto ideale. Quando si campiona il circuito, i campioni vengono prelevati da più istanze casuali, anziché da una sola.

![Rappresentazione del twirling di Pauli](../docs/images/guides/error-mitigation-explanation/pauli_twirling.avif)

Poiché la maggior parte degli errori nell'hardware quantistico attuale proviene dai gate a due qubit, questa tecnica viene spesso applicata esclusivamente ai gate nativi a due qubit. Il diagramma seguente illustra alcuni twirl di Pauli per i gate CNOT ed ECR. Ogni circuito all'interno di una riga ha lo stesso effetto ideale.

![Rappresentazione dei twirl dei gate](../docs/images/guides/error-mitigation-explanation/gate_twirls.avif)

Il twirling di Pauli può essere abilitato impostando `enable_gates` a `True` nelle [opzioni di twirling](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-twirling-options). Altre opzioni degne di nota includono:

- `num_randomizations`: Il numero di istanze di circuito da estrarre dall'insieme dei circuiti con twirling.
- `shots_per_randomization`: Il numero di shot da campionare da ciascuna istanza di circuito.

La seguente cella di codice mostra come abilitare il twirling di Pauli e impostare queste opzioni per l'estimator. Nessuna di queste opzioni deve essere impostata esplicitamente.

In [3]:
estimator = Estimator(mode=backend)
estimator.options.twirling.enable_gates = True
estimator.options.twirling.num_randomizations = 32
estimator.options.twirling.shots_per_randomization = 100

## Estinzione degli errori di readout con twirling (TREX)
La [Twirled Readout Error eXtinction (TREX)](https://journals.aps.org/pra/abstract/10.1103/PhysRevA.105.032620) mitiga l'effetto degli errori di misura per la stima dei valori di aspettazione di osservabili di Pauli.
Si basa sul concetto di misure con twirling, che vengono realizzate sostituendo casualmente i gate di misura con una sequenza composta da (1) un gate Pauli X, (2) una misura, e (3) un flip del bit classico. Proprio come nel twirling standard dei gate, questa sequenza equivale a una misura semplice in assenza di rumore, come illustrato nel diagramma seguente:

![Rappresentazione del twirling della misura](../docs/images/guides/error-mitigation-explanation/measurement_twirling.avif)

In presenza di errori di readout, il twirling della misura ha l'effetto di diagonalizzare la matrice di trasferimento degli errori di readout, rendendone facile l'inversione. La stima della matrice di trasferimento degli errori di readout richiede l'esecuzione di circuiti di calibrazione aggiuntivi, il che introduce un piccolo overhead.

TREX può essere abilitato impostando `measure_mitigation` a `True` nelle [opzioni di resilienza di Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) per Estimator. Le opzioni per l'apprendimento del rumore di misura sono descritte [qui](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-measure-noise-learning-options). Come per il twirling dei gate, puoi impostare il numero di randomizzazioni del circuito e il numero di shot per randomizzazione.

La seguente cella di codice mostra come abilitare TREX e impostare queste opzioni per l'estimator. Nessuna di queste opzioni deve essere impostata esplicitamente.

In [4]:
estimator = Estimator(mode=backend)
estimator.options.resilience.measure_mitigation = True
estimator.options.resilience.measure_noise_learning.num_randomizations = 32
estimator.options.resilience.measure_noise_learning.shots_per_randomization = 100

<span id="zne"></span>
## Estrapolazione a rumore zero (ZNE)
L'estrapolazione a rumore zero (ZNE, Zero-Noise Extrapolation) è una tecnica per mitigare gli errori nella stima dei valori di aspettazione di osservabili. Pur migliorando spesso i risultati, non garantisce di produrre un risultato privo di bias.

La ZNE è composta da due fasi:

1. _Amplificazione del rumore_: Il circuito quantistico originale viene eseguito più volte a diversi livelli di rumore.
2. _Estrapolazione_: Il risultato ideale viene stimato estrapolando i risultati dei valori di aspettazione rumorosi al limite di rumore zero.

Sia la fase di amplificazione del rumore che quella di estrapolazione possono essere implementate in molti modi diversi. Qiskit Runtime implementa l'amplificazione del rumore tramite il "gate folding digitale", ovvero i gate a due qubit vengono sostituiti con sequenze equivalenti del gate e del suo inverso. Per esempio, sostituire un unitario $U$ con $U U^\dagger U$ produce un fattore di amplificazione del rumore pari a 3. Per l'estrapolazione, puoi scegliere tra diverse forme funzionali, tra cui un fit lineare o un fit esponenziale.
L'immagine qui sotto illustra il gate folding digitale a sinistra e la procedura di estrapolazione a destra.

![Rappresentazione della ZNE](../docs/images/guides/error-mitigation-explanation/zne.avif)

La ZNE può essere abilitata impostando `zne_mitigation` a `True` nelle [opzioni di resilienza di Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) per Estimator.
Le opzioni di Qiskit Runtime per la ZNE sono descritte [qui](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options). Le seguenti opzioni sono degne di nota:

- `noise_factors`: I fattori di rumore da usare per l'amplificazione del rumore.
- `extrapolator`: La forma funzionale da usare per l'estrapolazione.

La seguente cella di codice mostra come abilitare la ZNE e impostare queste opzioni per l'estimator. Nessuna di queste opzioni deve essere impostata esplicitamente.

In [5]:
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.noise_factors = (1, 3, 5)
estimator.options.resilience.zne.extrapolator = "exponential"

<span id="pea"></span>
## Amplificazione probabilistica degli errori (PEA)
Una delle principali sfide della ZNE è amplificare con precisione il rumore che affligge il circuito target. Il gate folding fornisce un modo semplice per eseguire questa amplificazione, ma è potenzialmente impreciso e potrebbe portare a risultati errati. Consulta l'articolo ["Scalable error mitigation for noisy quantum circuits produces competitive expectation values"](https://arxiv.org/pdf/2108.09197), e in particolare la pagina 4 delle informazioni supplementari per i dettagli. L'amplificazione probabilistica degli errori fornisce un approccio più preciso all'amplificazione degli errori tramite l'apprendimento del rumore.

La PEA è una tecnica più sofisticata che esegue esperimenti preliminari per ricostruire il rumore e poi utilizza queste informazioni per eseguire un'amplificazione accurata. Inizia apprendendo il modello di rumore con twirling di ogni strato di gate di entanglement nel circuito prima della loro esecuzione (vedi [LayerNoiseLearningOptions](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-layer-noise-learning-options) per le opzioni di apprendimento rilevanti). Dopo la fase di apprendimento, i circuiti vengono eseguiti a ciascun fattore di rumore, dove ogni strato di entanglement dei circuiti viene amplificato iniettando in modo probabilistico rumore a singolo qubit proporzionale al corrispondente modello di rumore appreso. Consulta l'articolo ["Evidence for the utility of quantum computing before fault tolerance"](https://www.nature.com/articles/s41586-023-06096-3) per maggiori dettagli.

La PEA è composta da tre fasi:
1. _Apprendimento_: Viene appreso il modello di rumore con twirling di ogni strato di gate di entanglement nel circuito.
1. _Amplificazione del rumore_: Il circuito quantistico originale viene eseguito più volte a diversi fattori di rumore.
2. _Estrapolazione_: Il risultato ideale viene stimato estrapolando i risultati dei valori di aspettazione rumorosi al limite di rumore zero.

Per esperimenti su scala utility, la PEA è spesso la scelta migliore.

Poiché la PEA è una tecnica di amplificazione del rumore per la ZNE, devi anche abilitare la ZNE impostando `resilience.zne_mitigation = True`. Le altre opzioni di [`resilience.zne`](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-zne-options) possono essere usate in aggiunta per impostare gli estrapolatori, i livelli di amplificazione e così via. La PEA richiede un modello di rumore, che viene generato automaticamente quando si usano le primitive.

Il seguente snippet fornisce un esempio in cui la PEA viene usata per mitigare il risultato di un job dell'Estimator:

In [6]:
estimator = Estimator(mode=backend)
estimator.options.resilience.zne_mitigation = True
estimator.options.resilience.zne.amplifier = "pea"

<span id="pec"></span>
## Cancellazione probabilistica degli errori (PEC)
La cancellazione probabilistica degli errori (PEC, Probabilistic Error Cancellation) è una tecnica per mitigare gli errori nella stima dei valori di aspettazione di osservabili. A differenza della ZNE, restituisce una stima non distorta del valore di aspettazione. Tuttavia, in generale comporta un overhead maggiore.

Nella PEC, l'effetto di un circuito target ideale è espresso come combinazione lineare di circuiti rumorosi effettivamente implementabili in pratica:

$$
\mathcal{O}_{\text{ideal}} = \sum_{i} \eta_i \mathcal{O}_{noisy, i}
$$

L'output del circuito ideale può quindi essere riprodotto eseguendo diverse istanze di circuito rumoroso estratte da un insieme casuale definito dalla combinazione lineare. Se i coefficienti $\eta_i$ formano una distribuzione di probabilità, possono essere usati direttamente come probabilità dell'insieme. In pratica, alcuni coefficienti sono negativi, quindi formano invece una distribuzione di quasi-probabilità. Possono comunque essere usati per definire un insieme casuale, ma esiste un overhead di campionamento legato alla negatività della distribuzione di quasi-probabilità, che è caratterizzato dalla quantità

$$
\gamma = \sum_{i} \lvert \eta_i \rvert \geq 1.
$$

L'overhead di campionamento è un fattore moltiplicativo sul numero di shot necessari per stimare un valore di aspettazione con una data precisione, rispetto al numero di shot che sarebbero necessari dal circuito ideale. Scala in modo quadratico con $\gamma$, che a sua volta scala in modo esponenziale con la profondità del circuito.

La PEC può essere abilitata impostando `pec_mitigation` a `True` nelle [opzioni di resilienza di Qiskit Runtime](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-resilience-options-v2) per Estimator.
Le opzioni di Qiskit Runtime per la PEC sono descritte [qui](https://docs.quantum.ibm.com/api/qiskit-ibm-runtime/options-pec-options). È possibile impostare un limite sull'overhead di campionamento usando l'opzione `max_overhead`. Tieni presente che limitare l'overhead di campionamento può far sì che la precisione del risultato superi la precisione richiesta. Il valore predefinito di `max_overhead` è 100.

La seguente cella di codice mostra come abilitare la PEC e impostare l'opzione `max_overhead` per l'estimator.

In [7]:
estimator = Estimator(mode=backend)
estimator.options.resilience.pec_mitigation = True
estimator.options.resilience.pec.max_overhead = 100

## Passi successivi
- Consulta il [tutorial](/tutorials/combine-error-mitigation-techniques) sulla combinazione delle opzioni di mitigazione degli errori con la primitiva Estimator.
- [Configura la mitigazione degli errori.](configure-error-mitigation)
- [Configura la soppressione degli errori.](configure-error-suppression)
- Esplora le altre [opzioni](runtime-options-overview) per le primitive di Qiskit Runtime.
- Decidi in quale [modalità di esecuzione](execution-modes) eseguire il tuo job.